# 03 - Modèle ALS & évaluation
Ce notebook lit `evaluation_report.json` pour récupérer les meilleurs hyperparamètres, réentraîne ALS et illustre les métriques + recommandations.

## 1) Chargement des données prêtes ALS
On utilise `data/processed/events_clean` (export Parquet).

In [ ]:
import json
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml.recommendation import ALS

spark = SparkSession.builder.appName('ShopAI-03-ModelEvaluation').getOrCreate()
spark.sparkContext.setLogLevel('ERROR')

events = spark.read.parquet('data/processed/events_clean')
print('Lignes events_clean:', events.count())
events.select('visitorid','itemid','event','weight').show(5, truncate=False)


## 2) Charger le meilleur modèle
Le script `train_als.py` doit avoir généré `data/models/evaluation_report.json`.

In [ ]:
report_path = 'data/models/evaluation_report.json'
assert os.path.exists(report_path), f'report absent: {report_path}'
with open(report_path, 'r', encoding='utf-8') as f:
    report = json.load(f)
best = report['best_model']
best_params = best['hyperparams']
metrics = best['metrics']
print('Meilleur hyperparams:', best_params)
print('Métriques best:', metrics)


## 3) Préparer le DataFrame (userId, itemId, rating)
Retailrocket -> userId=visitorid, itemId=itemid, rating=weight

In [ ]:
retailrocket_als = events.select(
    col('visitorid').alias('userId').cast('int'),
    col('itemid').alias('itemId').cast('int'),
    col('weight').alias('rating').cast('float'),
)
.dropna()

retailrocket_als.groupBy('rating').count().show(5, truncate=False)
print('Interactions pour ALS:', retailrocket_als.count())


## 4) Split train/test (approx)
Pour le notebook, on effectue un split rapide par aléa. Le split stratifié complet est implémenté dans `train_als.py`.

In [ ]:
from pyspark.sql.functions import rand
df = retailrocket_als
df = df.withColumn('r', rand(42))
train_df = df.where(col('r') < 0.8).drop('r')
test_df  = df.where(col('r') >= 0.8).drop('r')
print('train:', train_df.count(), 'test:', test_df.count())


## 5) Entraînement ALS


In [ ]:
als = ALS(
    userCol='userId',
    itemCol='itemId',
    ratingCol='rating',
    rank=int(best_params['rank']),
    regParam=float(best_params['regParam']),
    maxIter=int(best_params['maxIter']),
    coldStartStrategy='drop',
)
model = als.fit(train_df)
print('Modèle entraîné ✅')


## 6) Évaluation (evaluate_model)
On utilise `evaluate_model` depuis `src/models/evaluate.py`.

In [ ]:
import sys
sys.path.append('src')
from models.evaluate import evaluate_model

k = 10
eval_metrics = evaluate_model(model, test_df, k=k)
print(eval_metrics)


## 7) Visualisation des métriques


In [ ]:
import matplotlib.pyplot as plt
labels = ['RMSE','Precision@10','Recall@10','NDCG@10']
values = [
    eval_metrics['rmse'],
    eval_metrics['precision_at_k'],
    eval_metrics['recall_at_k'],
    eval_metrics['ndcg_at_k'],
]
plt.figure(figsize=(7,4))
plt.bar(labels, values)
plt.title('Métriques du meilleur modèle (sur ce split)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 8) Recommandations pour 5 utilisateurs


In [ ]:
from pyspark.sql import Row
user_sample = [r for r in test_df.select('userId').distinct().limit(5).collect()]
user_ids = [int(r['userId']) for r in user_sample]
user_df = spark.createDataFrame([Row(userId=u) for u in user_ids])
recs = model.recommendForUserSubset(user_df, 10).collect()
for row in recs:
    uid = row['userId']
    print('User', uid)
    for rec in row['recommendations']:
        print('  item:', rec['itemId'], 'score:', float(rec['rating']))


## 9) Distribution des scores recommandés


In [ ]:
scores = []
for row in recs:
    for rec in row['recommendations']:
        scores.append(float(rec['rating']))

import seaborn as sns
plt.figure(figsize=(7,4))
sns.histplot(scores, bins=20, kde=True)
plt.title('Distribution des scores recommandés (échantillon)')
plt.tight_layout()
plt.show()
